# COLLECT THAI PREDICTION DATASET
Run in Google Colab and export data into Google Drive

## GEE Setup

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project=PROJECT_NAME)

In [ ]:
import pandas as pd
from datetime import datetime

# Positive group

## Get dates

In [ ]:
def filter(i_date, f_date, cloud_coverage=20, spatial_coverage=50, bands=['B2', 'B3', 'B4', 'B8A', 'B11', 'B12'], tiles_list=None):
  hlss30 = ee.ImageCollection('NASA/HLS/HLSS30/v002')
  if tiles_list is None:
    tiles_list = ['47QLB', '47QMB', '47QNB', '47QPB', '47QQB',
                  '47QLA', '47QMA', '47QNA', '47QPA', '47QQA',
                  '47QMV', '47QNV']
  hlss30_filter = hlss30.filter(ee.Filter.inList('MGRS_TILE_ID', tiles_list))
  hlss30_filter = hlss30_filter.filterDate(i_date, f_date)
  hlss30_filter = hlss30_filter.filter(ee.Filter.lte('CLOUD_COVERAGE', cloud_coverage))
  hlss30_filter = hlss30_filter.filter(ee.Filter.gte('SPATIAL_COVERAGE', spatial_coverage))
  hlss30_filter = hlss30_filter.select(bands)
  return hlss30_filter

In [ ]:
def get_pos_date(year, tile_list):
  # Febuary to April
  i_date = f"{year}-01"
  f_date = f"{year}-05"
  ee_tile_list = ee.List(tile_list)
  count = 0

  # Filter and sampling from i_date and f_date
  sampled_collection = filter(i_date=i_date, f_date=f_date)
  count += sampled_collection.size().getInfo()

  # Add date from timestamp
  def add_date(image):
      date_str = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
      return image.set('date', date_str)
  sampled_collection = sampled_collection.map(add_date)

  # Get tiles on each date
  def tiles_each_date(date_str):
    images = sampled_collection.filter(ee.Filter.eq('date', date_str))
    tile_ids = ee.List(images.aggregate_array('MGRS_TILE_ID').distinct())

    tile_flags = ee.Dictionary.fromLists(
        ee_tile_list,
        ee_tile_list.map(lambda tile: tile_ids.contains(tile))
    )

    return tile_flags.set('date', date_str)

  unique_dates = sampled_collection.aggregate_array('date').distinct()
  results = unique_dates.map(tiles_each_date)
  output = results.getInfo()
  df = pd.DataFrame(output)

  return df

In [ ]:
tile_list = ['47QLB', '47QMB', '47QNB', '47QPB', '47QQB',
             '47QLA', '47QMA', '47QNA', '47QPA', '47QQA',
             '47QMV', '47QNV']

In [ ]:
# 2024
pos_2024 = get_pos_date(year=2024, tile_list=tile_list)
pos_2024.to_csv('pos_2024.csv', index=False)
pos_2024.shape

In [ ]:
# 2023
pos_2023 = get_pos_date(year=2023, tile_list=tile_list)
pos_2023.to_csv('pos_2023.csv', index=False)
pos_2023.shape

In [ ]:
# 2022
pos_2022 = get_pos_date(year=2022, tile_list=tile_list)
pos_2022.to_csv('pos_2022.csv', index=False)
pos_2022.shape

In [ ]:
# 2021
pos_2021 = get_pos_date(year=2021, tile_list=tile_list)
pos_2021.to_csv('pos_2021.csv', index=False)
pos_2021.shape

## Get tiles according hotspot with sampling

In [ ]:
hotspot_df = pd.read_csv("hotspot.csv")
tile_list = ['47QLB', '47QMB', '47QNB', '47QPB', '47QQB',
             '47QLA', '47QMA', '47QNA', '47QPA', '47QQA',
             '47QMV', '47QNV']
hotspot_df[tile_list] = hotspot_df[tile_list].astype(bool)
hotspot_df['tile_selected'] = hotspot_df[tile_list].apply(lambda row: [col for col in row.index if row[col]], axis=1)
hotspot_df.head()

In [ ]:
def get_ids_list(i_date, f_date, tile_id):
  def filter(i_date, f_date, cloud_coverage=20, spatial_coverage=50, bands=['B2', 'B3', 'B4', 'B8A', 'B11', 'B12'], tiles_list=None):
    hlss30 = ee.ImageCollection('NASA/HLS/HLSS30/v002')
    if tiles_list is None:
      tiles_list = ['47QLB', '47QMB', '47QNB', '47QPB', '47QQB',
                    '47QLA', '47QMA', '47QNA', '47QPA', '47QQA',
                    '47QMV', '47QNV']
    hlss30_filter = hlss30.filter(ee.Filter.inList('MGRS_TILE_ID', tiles_list))
    hlss30_filter = hlss30_filter.filterDate(i_date, f_date)
    hlss30_filter = hlss30_filter.filter(ee.Filter.lte('CLOUD_COVERAGE', cloud_coverage))
    hlss30_filter = hlss30_filter.filter(ee.Filter.gte('SPATIAL_COVERAGE', spatial_coverage))
    hlss30_filter = hlss30_filter.select(bands)
    return hlss30_filter

  collection = filter(i_date, f_date, tiles_list=tile_id)
  id_list = collection.aggregate_array("system:id").getInfo()
  id_list = [s.replace("NASA/HLS/HLSS30/v002/", "") for s in id_list]
  return id_list

In [ ]:
id_list = []
for i in range(len(hotspot_df)):
  i_date = hotspot_df.iloc[i]["i_date"]
  f_date = hotspot_df.iloc[i]["f_date"]
  tile_selected = hotspot_df.iloc[i]["tile_selected"]
  id_list += get_ids_list(i_date, f_date, tile_selected)
len(id_list)

In [ ]:
sorted(id_list)

In [ ]:
burn_df = pd.DataFrame(id_list, columns=['ID'])
burn_df["MGRS_Tile"] = burn_df["ID"].str.split('_').str[0]
burn_df["Year"] = burn_df["ID"].str.slice(7, 11)
burn_df["Month"] = burn_df["ID"].str.slice(11, 13)
burn_df.head()

In [ ]:
burn_df[["ID", "Year"]].groupby("Year").count()

In [ ]:
burn_df[["ID", "Month"]].groupby("Month").count()

In [ ]:
burn_df.to_csv("burn_tiles.csv")

# Negative group

In [ ]:
def filter(i_date, f_date, cloud_coverage=20, spatial_coverage=50, bands=['B2', 'B3', 'B4', 'B8A', 'B11', 'B12'], tiles_list=None):
  hlss30 = ee.ImageCollection('NASA/HLS/HLSS30/v002')
  if tiles_list is None:
    tiles_list = ['47QLB', '47QMB', '47QNB', '47QPB', '47QQB',
                  '47QLA', '47QMA', '47QNA', '47QPA', '47QQA',
                  '47QMV', '47QNV']
  hlss30_filter = hlss30.filter(ee.Filter.inList('MGRS_TILE_ID', tiles_list))
  hlss30_filter = hlss30_filter.filterDate(i_date, f_date)
  hlss30_filter = hlss30_filter.filter(ee.Filter.lte('CLOUD_COVERAGE', cloud_coverage))
  hlss30_filter = hlss30_filter.filter(ee.Filter.gte('SPATIAL_COVERAGE', spatial_coverage))
  hlss30_filter = hlss30_filter.select(bands)
  return hlss30_filter

In [ ]:
neg_2021 = filter(i_date='2021-07-01', f_date='2021-12-31', cloud_coverage=30)
neg_2022 = filter(i_date='2022-07-01', f_date='2022-12-31', cloud_coverage=30)
neg_2023 = filter(i_date='2023-07-01', f_date='2023-12-31', cloud_coverage=30)
neg_2024 = filter(i_date='2024-07-01', f_date='2024-12-31', cloud_coverage=30)
print(f"Total negative at 2021: {neg_2021.size().getInfo()}")
print(f"Total negative at 2022: {neg_2022.size().getInfo()}")
print(f"Total negative at 2023: {neg_2023.size().getInfo()}")
print(f"Total negative at 2024: {neg_2024.size().getInfo()}")

In [ ]:
def sampling_image(image_collection, n_sample):
  tile_list = ['47QLB', '47QMB', '47QNB', '47QPB', '47QQB',
               '47QLA', '47QMA', '47QNA', '47QPA', '47QQA',
               '47QMV', '47QNV']

  image_collection = image_collection.randomColumn('random', distribution='normal')
  sampled_images = []
  for tile in tile_list:
    filtered = image_collection.filter(ee.Filter.eq('MGRS_TILE_ID', tile)).sort('random')
    n = min(n_sample, filtered.size().getInfo())
    # print(f"Sample {tile} with {n} patches")
    for i in range(n):
        image = ee.Image(filtered.toList(n_sample).get(i))
        sampled_images.append(image)
  sampled_collection = ee.ImageCollection.fromImages(sampled_images)

  id_list = sampled_collection.aggregate_array("system:id").getInfo()
  id_list = [s.replace("NASA/HLS/HLSS30/v002/", "") for s in id_list]

  return id_list

In [ ]:
neg_2021_sampled = sampling_image(neg_2021, 2)
neg_2022_sampled = sampling_image(neg_2022, 2)
neg_2023_sampled = sampling_image(neg_2023, 2)
neg_2024_sampled = sampling_image(neg_2024, 2)

nonburn_tiles = neg_2021_sampled + neg_2022_sampled + neg_2023_sampled + neg_2024_sampled
len(nonburn_tiles)

In [ ]:
unburn_df = pd.DataFrame(nonburn_tiles, columns=['ID'])
unburn_df["MGRS_Tile"] = unburn_df["ID"].str.split('_').str[0]
unburn_df["Year"] = unburn_df["ID"].str.slice(7, 11)
unburn_df["Month"] = unburn_df["ID"].str.slice(11, 13)
unburn_df.info()

In [ ]:
unburn_df[["ID", "Year"]].groupby("Year").count()

In [ ]:
unburn_df[["ID", "Month"]].groupby("Month").count()

In [ ]:
unburn_df.to_csv("unburn_tiles.csv")

# Merge positive and negative group

In [ ]:
unburn_df = pd.read_csv("unburn_tiles.csv")
unburn_df = unburn_df.iloc[:, 1:]
unburn_df.rename(columns={"ID": "Unburned ID"}, inplace=True)
unburn_df.drop(columns=["Month"], inplace=True)
unburn_df.info()

In [ ]:
unburn_df["group_id"] = (
    unburn_df
    .groupby(["Year", "MGRS_Tile"])
    .cumcount() + 1
)
unburn_df.sort_values(by=["Year", "MGRS_Tile"])

In [ ]:
burn_df = pd.read_csv("burn_tiles.csv")
burn_df = burn_df.iloc[:, 1:]
burn_df.rename(columns={"ID": "Burned ID"}, inplace=True)
burn_df.drop(columns=["Month"], inplace=True)
burn_df.info()

In [ ]:
burn_df["group_id"] = (
    burn_df
    .groupby(["Year", "MGRS_Tile"])
    .cumcount() + 1
)
burn_df.sort_values(by=["Year", "MGRS_Tile"])

In [ ]:
merge_df = pd.merge(burn_df, unburn_df, on=["MGRS_Tile", "Year", "group_id"], how="inner")
merge_df.info()

In [ ]:
merge_df = merge_df[["MGRS_Tile", "Year", "Burned ID", "Unburned ID"]]
merge_df.rename(columns={"Burned ID": "Burned_ID", "Unburned ID": "Unburned_ID"}, inplace=True)
merge_df.info()

In [ ]:
merge_df.to_csv("index.csv", index=False)

# Visualization example

In [ ]:
merge_df = pd.read_csv("index.csv")
merge_df = merge_df.iloc[:, 1:]
merge_df.head(20)

In [ ]:
all_ids = merge_df["Burned_ID"].tolist() + merge_df["Unburned_ID"].tolist()
all_ids = list(set(all_ids))
all_ids = ["NASA/HLS/HLSS30/v002/" + a for a in all_ids]

In [ ]:
hlss30 = ee.ImageCollection('NASA/HLS/HLSS30/v002')
hlss30_filter = hlss30.filter(ee.Filter.inList("system:id", all_ids))
hlss30_filter.size().getInfo()

In [ ]:
def normalize_band(image):
    min_list = [0.01, 0.01, 0.01, 0.01, 0.005, 0.005]
    max_list = [0.18, 0.18, 0.18, 0.5, 0.35, 0.3]

    band_names = ["B2", "B3", "B4", "B8A", "B11", "B12"]
    normalized_bands = []
    for band, min_val, max_val in zip(band_names, min_list, max_list):
        band_img = image.select(band)
        norm_band = band_img.subtract(min_val).divide(max_val - min_val).clamp(0,1).rename(band)
        normalized_bands.append(norm_band)

    normalized_image = ee.Image(normalized_bands)
    normalized_image = normalized_image.unmask(0)

    return normalized_image

In [ ]:
def create_layer(image, title, bands):
  vis_params = {
      'min': 0,
      'max':1,
      'bands': bands
  }
  layer = geemap.ee_tile_layer(image, vis_params, name=title)

  return layer

In [ ]:
merge_df[merge_df["Year"]==2024].sort_values(by=["Burned_ID"])

In [ ]:
def plot_compare(merge_df, index, bands=["B4", "B3", "B2"]):
  burn_id = "NASA/HLS/HLSS30/v002/" + merge_df.iloc[index]["Burned_ID"]
  unburn_id = "NASA/HLS/HLSS30/v002/" + merge_df.iloc[index]["Unburned_ID"]
  mgrs_tile = merge_df.iloc[index]["Burned_ID"].split("_")[0][1:]
  print(f"MGRS tile: {mgrs_tile}")
  print(f"Burn id: {burn_id}")
  print(f"Unburn id: {unburn_id}")

  # Get feature collection for MGRS tile grid
  center_long, center_lat, geometry = get_geometry(mgrs_tile)
  fc = create_mgrs_patch(mgrs_code=mgrs_tile, center_long=center_long, center_lat=center_lat, split=True)

  hlss30 = ee.ImageCollection('NASA/HLS/HLSS30/v002')
  hlss30 = hlss30.select(["B2", "B3", "B4", "B8A", "B11", "B12"])
  hlss30_burn = hlss30.filter(ee.Filter.eq("system:id", burn_id))
  hlss30_unburn = hlss30.filter(ee.Filter.eq("system:id", unburn_id))
  bounds  = hlss30_burn.geometry().bounds()
  center = bounds.centroid(maxError=100).coordinates().getInfo()

  normalized_burn = normalize_band(hlss30_burn.first())
  normalized_unburn = normalize_band(hlss30_unburn.first())

  burn_layer = create_layer(image=normalized_burn, title="Positive", bands=bands)
  nonburn_layer = create_layer(image=normalized_unburn, title="Negative", bands=bands)

  Map = geemap.Map(center=(center[1], center[0]), zoom=9)
  Map.split_map(nonburn_layer, burn_layer)
  Map.addLayer(fc, {'color': 'red'}, 'MGRS tiles')

  return Map

In [ ]:
rgb_bands = ['B4', 'B3', 'B2']
if_bands = ['B8A', 'B4', 'B3']
swir_bands = ['B12', 'B8A', 'B4']
agri_bands = ['B11', 'B8A', 'B2']

In [ ]:
# Tile: 47QMV (1)
map = plot_compare(merge_df, 76, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 44, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 45, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 31, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 32, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 33, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 24, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 25, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 26, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 27, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 28, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 29, rgb_bands)
map

In [ ]:
map = plot_compare(merge_df, 30, rgb_bands)
map

# Export Image

In [ ]:
hlss30 = ee.ImageCollection('NASA/HLS/HLSS30/v002')
hlss30 = hlss30.select(["B2", "B3", "B4", "B8A", "B11", "B12"])
burn_id = ["NASA/HLS/HLSS30/v002/" + a for a in merge_df["Burned_ID"]]
unburn_id = ["NASA/HLS/HLSS30/v002/" + a for a in merge_df["Unburned_ID"]]

hlss30_burn = hlss30.filter(ee.Filter.inList("system:id", burn_id))
hlss30_unburn = hlss30.filter(ee.Filter.inList("system:id", unburn_id))
print(f"Burned size: {hlss30_burn.size().getInfo()}")
print(f"Unburned size: {hlss30_unburn.size().getInfo()}")

In [ ]:
def export(image_collection, name, folder):
  def export_image(image, name, folder):
      no_data_val = 0.0
      image = image.unmask(no_data_val)
      export_task = ee.batch.Export.image.toDrive(
          image = image.toFloat(),
          description = f"{name}_{image.get('id').getInfo()}",
          folder = folder,
          fileNamePrefix = image.get('id').getInfo(),
          fileFormat = 'GeoTIFF',
          region = image.geometry(),
          scale=30,
          formatOptions={'noData': no_data_val},
      )
      export_task.start()

  def normalize_band(image):
    min_list = [0.01, 0.01, 0.01, 0.01, 0.005, 0.005]
    max_list = [0.18, 0.18, 0.18, 0.5, 0.35, 0.3]

    # Normalize
    band_names = ["B2", "B3", "B4", "B8A", "B11", "B12"]
    normalized_bands = []
    for band, min_val, max_val in zip(band_names, min_list, max_list):
        band_img = image.select(band)
        norm_band = band_img.subtract(min_val).divide(max_val - min_val)
        normalized_bands.append(norm_band)

    # Combine to img
    normalized_image = ee.Image.cat(normalized_bands).rename(band_names)
    normalized_image = normalized_image.clamp(0,1)
    normalized_image = normalized_image.unmask(0.0)
    normalized_image = normalized_image.set("id", image.id())

    return normalized_image

  # Exports
  bands = ["B2", "B3", "B4", "B8A", "B11", "B12"]
  for index, image in enumerate(image_collection.toList(image_collection.size()).getInfo()):
    image_ee = ee.Image(image['id']).select(bands)
    image_ee_normalized = normalize_band(image_ee)
    export_image(image_ee_normalized, name, folder)

In [ ]:
export(image_collection=hlss30_burn, name="Pos", folder=POSITIVE_FOLDER_NAME)

In [ ]:
export(image_collection=hlss30_unburn, name="Neg", folder=NEGATIVE_FOLDER_NAME)